In [ ]:
# Module 1 - Lab 1: Memory-Based vs Generator-Based Data Loading

import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import os

# ---------------- Memory-Based Loading ----------------
print("=== Memory-Based Loading ===")
# Example: Load small dataset into RAM
def load_images_to_memory(data_dir, img_size=(128,128)):
    images, labels = [], []
    for class_name in os.listdir(data_dir):
        class_dir = os.path.join(data_dir, class_name)
        if not os.path.isdir(class_dir): continue
        for img_name in os.listdir(class_dir):
            img_path = os.path.join(class_dir, img_name)
            img = plt.imread(img_path)
            img = np.resize(img, (*img_size, 3))  # Resize
            images.append(img)
            labels.append(class_name)
    return np.array(images), np.array(labels)

# Usage example:
# X_train, y_train = load_images_to_memory('train_data')

# ---------------- Generator-Based Loading (Keras) ----------------
print("\n=== Generator-Based (Keras) ===")
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    'train_data',
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

# ---------------- PyTorch Dataset + DataLoader ----------------
class CustomImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = os.listdir(root_dir)
        self.image_paths = []
        self.labels = []
        for label, class_name in enumerate(self.classes):
            class_dir = os.path.join(root_dir, class_name)
            for img_name in os.listdir(class_dir):
                self.image_paths.append(os.path.join(class_dir, img_name))
                self.labels.append(label)
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        image = plt.imread(self.image_paths[idx])
        if self.transform:
            image = self.transform(image)
        return image, self.labels[idx]

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset = CustomImageDataset('train_data', transform=transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

print("All three approaches ready!")

In [ ]:
# Module 1 - Lab 2: Data Loading + Augmentation with Keras

from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt

# Strong augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.3,
    height_shift_range=0.3,
    shear_range=0.2,
    zoom_range=0.3,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.7, 1.3],
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    'path/to/train',
    target_size=(224, 224),   # Common size for transfer learning
    batch_size=32,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    'path/to/validation',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

# Visualize augmentations
def plot_augmented_images(generator):
    imgs, labels = next(generator)
    plt.figure(figsize=(12, 8))
    for i in range(9):
        plt.subplot(3, 3, i+1)
        plt.imshow(imgs[i])
        plt.axis('off')
    plt.show()

plot_augmented_images(train_generator)

In [ ]:
# Module 1 - Lab 3: Data Loading + Augmentation with PyTorch

import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# Advanced transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(30),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Datasets
train_dataset = datasets.ImageFolder('path/to/train', transform=train_transform)
val_dataset = datasets.ImageFolder('path/to/validation', transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

print(f"Training samples: {len(train_dataset)} | Classes: {train_dataset.classes}")
